# The competition's evaluator loop on a Colab GPU (spec 028 AI9, #687)

**Runtime -> Change runtime type -> A100 GPU** (an L4 also works). Then **Runtime -> Run all**.
With Colab Pro+ you can close the tab: it keeps running (up to 24 hours).

**Once, before Run all** (already done if you ran `colab_agent.ipynb` before): the key icon on the left (Secrets), two
secrets with *Notebook access* on: `KAGGLE_USERNAME` = `jcblearning`, and `KAGGLE_API_TOKEN` = your Kaggle API token
(kaggle.com -> Settings -> API -> Generate New Token). Never paste the token into a cell.

What it does: installs the model server and the agent framework at the versions the Kaggle kernels used, puts the
competition's wheels, the model and one private task bundle (no fix, no tests) where Kaggle would mount them, and runs
the repository's own kernel file unchanged. The log and the output go to a private dataset
`jcblearning/htlnlp-colab-official-<label>` when it ends, however it ends, so nothing needs copying from this page.
Which runs it makes is read from `kaggle/colab_official_run.json` on the branch. **One tap is enough:** after the runs it started with, it keeps reading that list (every minute, for up to two and a half hours after each run) and makes any run added to it, with this branch's helper as it then is. So a full arm, whose plan is registered only after the test run has been read, starts without a second tap. When the list says stop, or names nothing new in that time, it gives the GPU back.


In [ ]:
import json, os, time, types, urllib.request
from google.colab import userdata
os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")
assert os.environ["KAGGLE_USERNAME"] and os.environ["KAGGLE_API_TOKEN"], "add both Colab secrets (see above)"

REPO = "healthtransformationlab/health-transformation-lab-nlp"
BRANCH = "feat/687-028-colab-paired-run"

def get(name):   # the branch's own file; the query keeps a cached copy from being served after a fix
    url = f"https://raw.githubusercontent.com/{REPO}/refs/heads/{BRANCH}/kaggle/{name}?t={int(time.time())}"
    return urllib.request.urlopen(url, timeout=60).read().decode()

co = types.ModuleType("colab_official")
exec(compile(get("colab_official.py"), "colab_official.py", "exec"), co.__dict__)
plan = json.loads(get("colab_official_run.json"))
print("runs:", [r["label"] for r in plan["runs"]])
for r in plan["runs"]:
    code = co.main(r["label"], r["bundle"], kernel_dataset=r.get("kernel_dataset"), kernel_url=r.get("kernel_url"))
    print(r["label"], "finished, kernel exit", code)
print("ALL DONE. Nothing needs copying: the log and results are on Kaggle.")
